# 🧠 Notebook 03 — Solver Agent (LLaMA-3-8B Fine-tuning)
### Multi-Agent System for Secure Clinical Summarization



**Runtime: A100 = ~6-8h | V100 = ~10-12h | T4 = ~18-24h **


In [ ]:
# ── Step 1: GPU Check ─────────────────────────────────────────────────────────
import torch

if not torch.cuda.is_available():
    raise RuntimeError(
        'No GPU detected. Go to Runtime > Change runtime type > GPU and restart.'
    )

gpu_name = torch.cuda.get_device_name(0)
gpu_mem  = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f'GPU  : {gpu_name}')
print(f'VRAM : {gpu_mem:.1f} GB')

# Adapt settings to available GPU
if 'A100' in gpu_name:
    N_TRAIN_SAMPLES = 200_000
    BATCH_SIZE      = 4
    GRAD_ACCUM      = 4
    DTYPE           = 'bfloat16'
    print('A100 detected - full settings (200K samples, bfloat16)')
elif 'V100' in gpu_name:
    N_TRAIN_SAMPLES = 50_000
    BATCH_SIZE      = 2
    GRAD_ACCUM      = 8
    DTYPE           = 'float16'
    print('V100 detected - reduced settings (50K samples, float16)')
else:
    # T4 or other
    N_TRAIN_SAMPLES = 20_000
    BATCH_SIZE      = 1
    GRAD_ACCUM      = 16
    DTYPE           = 'float16'
    print(f'{gpu_name} detected - minimum settings (20K samples, float16)')
    print('Training will take 18-24 hours. Colab Pro+ with A100 is strongly recommended.')

EFFECTIVE_BATCH = BATCH_SIZE * GRAD_ACCUM
print(f'\nEffective batch size : {EFFECTIVE_BATCH}')
print(f'Training samples     : {N_TRAIN_SAMPLES:,}')


GPU  : NVIDIA A100-SXM4-80GB
VRAM : 85.1 GB
A100 detected - full settings (200K samples, bfloat16)

Effective batch size : 16
Training samples     : 200,000


In [ ]:
# ── Step 2: Mount Drive and set paths ────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')
import os

BASE_DIR      = '/content/drive/MyDrive/clinical_mas'
DISCHARGE_CSV = f'{BASE_DIR}/data/raw/discharge.csv'
PROC_DIR      = f'{BASE_DIR}/data/processed'
TRAIN_CSV     = f'{PROC_DIR}/solver_train_data.csv'
CHECKPT_DIR  = f'{BASE_DIR}/models/solver_checkpoints'
AGENTS_DIR    = f'{BASE_DIR}/agents'

for d in [PROC_DIR, CHECKPT_DIR, AGENTS_DIR]:
    os.makedirs(d, exist_ok=True)

print('Drive mounted')
if not os.path.exists(DISCHARGE_CSV):
    print(f'ERROR: discharge.csv not found at {DISCHARGE_CSV}')
else:
    print(f'Found discharge.csv ({os.path.getsize(DISCHARGE_CSV)/1e9:.2f} GB)')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Drive mounted
Found discharge.csv (3.53 GB)


In [ ]:
# ── Step 3: Install dependencies ─────────────────────────────────────────────
# bitsandbytes: no version pin — 0.43.x is broken on Colab CUDA 12.x + Python 3.12
# triton: upgrade to ensure triton.ops is available for bitsandbytes CUDA kernels
!pip install -q -U bitsandbytes triton
!pip install -q transformers==4.44.0 peft==0.12.0 trl==0.11.0 accelerate==0.33.0
!pip install -q rouge-score
print('Done')


Done


In [ ]:
# ── Step 4: HuggingFace login ─────────────────────────────────────────────────
# Get token from: https://huggingface.co/settings/tokens
# Accept LLaMA-3 license at: https://huggingface.co/meta-llama/Meta-Llama-3-8B-Instruct

from huggingface_hub import login


from google.colab import userdata
HF_TOKEN = userdata.get('HF_TOKEN')


login(token=HF_TOKEN)
print('Logged in to HuggingFace')


Logged in to HuggingFace


## Step 5 — Extract BHC Training Pairs


**Input**: full note with BHC section removed
**Target**: BHC section only


In [ ]:
import re
import pandas as pd
import time

# ── BHC section header patterns ────────────────────────
# These match all common variations found in MIMIC-IV.
BHC_HEADER = re.compile(
    r'(?i)'
    r'(brief\s+hospital\s+course'
    r'|brief\s+hosp\.?\s+course'
    r'|hospital\s+course'
    r'|hosp\.?\s+course'
    r'|summary\s+of\s+hospital\s+course'
    r')'
    r'\s*:?\s*\n?',
)

# ── Next-section boundary pattern ─────────────────────────────────────────────
# A new section starts when a line has Title Case words followed by a colon.
# This is how MIMIC notes separate all sections.
NEXT_SECTION = re.compile(
    r'\n(?=[A-Z][A-Za-z0-9\s/\-\.]+:)',
    re.MULTILINE
)


def extract_bhc_pair(note_text):
    '''
    Extract (note_without_bhc, bhc_target) from a MIMIC-IV discharge note.

    Returns:
        Tuple (input_note, bhc_text) if a valid pair is found.
        None if the note does not have a usable BHC section.

    Quality filters applied:
        - BHC must be found
        - BHC: 50-400 words (short enough to be a summary, not another note)
        - Input note after BHC removal: >= 200 words (enough content to summarize)
    '''
    # Find BHC header
    header_match = BHC_HEADER.search(note_text)
    if not header_match:
        return None

    bhc_content_start = header_match.end()

    # Find where the next section begins after the BHC
    next_section_match = NEXT_SECTION.search(note_text, bhc_content_start)
    bhc_content_end = next_section_match.start() if next_section_match else len(note_text)

    bhc_text = note_text[bhc_content_start:bhc_content_end].strip()

    # Filter 1: BHC word count
    bhc_words = len(bhc_text.split())
    if bhc_words < 50:
        return None
    if bhc_words > 400:
        return None

    # Remove the entire BHC block from the note to create the input
    note_without_bhc = (
        note_text[:header_match.start()] +
        note_text[bhc_content_end:]
    ).strip()

    # Clean up triple-newlines left behind
    note_without_bhc = re.sub(r'\n{3,}', '\n\n', note_without_bhc)

    # Filter 2: remaining note word count
    if len(note_without_bhc.split()) < 200:
        return None

    return note_without_bhc, bhc_text


# ── Quick test on a synthetic example ────────────────────────────────────────
sample_note = '''Chief Complaint:
Shortness of breath and bilateral leg swelling.

History of Present Illness:
Mr. Jones is a 72-year-old male with a past medical history significant for
ischemic cardiomyopathy with an ejection fraction of 35%, hypertension, type 2
diabetes mellitus, and chronic kidney disease stage 3. He presented to the
emergency department with a 3-day history of progressively worsening dyspnea
on exertion, orthopnea requiring three pillows to sleep, paroxysmal nocturnal
dyspnea, and bilateral lower extremity edema. He denied any chest pain,
palpitations, or syncope. His weight had increased by 8 pounds over the past
week. He was last seen in cardiology clinic two months ago at which time his
medications were uptitrated. He reports good compliance with his medications
but admits to increased dietary sodium intake over the holiday period.

Past Medical History:
1. Ischemic cardiomyopathy, EF 35%
2. Hypertension
3. Type 2 diabetes mellitus
4. Chronic kidney disease stage 3, baseline creatinine 1.6

Physical Exam:
Vitals: BP 158/92, HR 98, RR 20, SpO2 89% on room air, weight 94kg (dry 86kg).
General: elderly male in mild-to-moderate respiratory distress.
Chest: bilateral rales to mid-lung fields bilaterally.
Cardiovascular: S3 gallop, JVD to the angle of the jaw.
Extremities: 3+ pitting edema to the knees bilaterally.

Pertinent Results:
BNP 2340 pg/mL. Creatinine 2.1 (baseline 1.6). Na 133. Chest X-ray with
cardiomegaly and bilateral pulmonary edema. Echo EF 35%, moderate mitral
regurgitation, elevated PCWP estimated 28 mmHg.

Brief Hospital Course:
Patient was admitted for acute decompensated heart failure. He was started on
intravenous furosemide 80mg twice daily with good diuretic response, losing
approximately 4 liters over the first two days. His respiratory status improved
significantly. He was transitioned to oral furosemide 40mg daily when clinically
stable. Cardiology was consulted and uptitrated his carvedilol and lisinopril.
He was discharged in stable condition with close outpatient follow-up arranged.

Discharge Medications:
1. Furosemide 40mg daily
2. Carvedilol 12.5mg twice daily
3. Lisinopril 10mg daily

Discharge Diagnosis:
1. Acute decompensated congestive heart failure
2. Hypertension

Followup Instructions:
Follow up with Cardiology in 1 week.
'''

result = extract_bhc_pair(sample_note)
if result:
    note_in, bhc_out = result
    print('=== BHC Extraction Test ===')
    print(f'BHC word count   : {len(bhc_out.split())} words')
    print(f'Input word count : {len(note_in.split())} words (after BHC removal)')
    print()
    print('BHC TARGET (what model learns to write):')
    print('-' * 60)
    print(bhc_out)
    print('-' * 60)
    print()
    print('Note: BHC is in clinical language.')
    print('At inference, the system prompt converts this to patient-friendly language.')
    print('The model learns the COMPRESSION task; the prompt controls the STYLE.')
else:
    print('ERROR: BHC extraction failed on test note')


=== BHC Extraction Test ===
BHC word count   : 55 words
Input word count : 278 words (after BHC removal)

BHC TARGET (what model learns to write):
------------------------------------------------------------
Patient was admitted for acute decompensated heart failure. He was started on
intravenous furosemide 80mg twice daily with good diuretic response, losing
approximately 4 liters over the first two days. His respiratory status improved
significantly. He was transitioned to oral furosemide 40mg daily when clinically
stable. Cardiology was consulted and uptitrated his carvedilol and lisinopril.
------------------------------------------------------------

Note: BHC is in clinical language.
At inference, the system prompt converts this to patient-friendly language.
The model learns the COMPRESSION task; the prompt controls the STYLE.


In [ ]:
# ── Step 6: Build the training dataset ────────────────────────────────────────
# Processes MIMIC-IV notes and extracts BHC pairs.

if os.path.exists(TRAIN_CSV):
    train_df = pd.read_csv(TRAIN_CSV)
    print(f'Loaded existing training data: {len(train_df):,} samples')
    print(f'Sample input (100 chars)  : "{train_df.iloc[0]["note_text"][:100]}"')
    print(f'Sample target (100 chars) : "{train_df.iloc[0]["bhc_text"][:100]}"')
else:
    load_rows = min(N_TRAIN_SAMPLES * 4, 500_000)
    print(f'Loading up to {load_rows:,} notes from discharge.csv...')

    df_raw = pd.read_csv(
        DISCHARGE_CSV,
        usecols=['subject_id', 'text'],
        nrows=load_rows,
        low_memory=False
    ).dropna(subset=['text'])

    print(f'Loaded {len(df_raw):,} notes')
    print(f'Extracting BHC pairs (target: {N_TRAIN_SAMPLES:,})...\n')

    pairs       = []
    rejected    = 0
    t0          = time.time()

    for i, (_, row) in enumerate(df_raw.iterrows()):
        if len(pairs) >= N_TRAIN_SAMPLES:
            break

        result = extract_bhc_pair(str(row['text']))
        if result is None:
            rejected += 1
            continue

        note_in, bhc_out = result
        pairs.append({
            'note_text': note_in,
            'bhc_text' : bhc_out,
        })

        if (i + 1) % 20_000 == 0:
            elapsed     = time.time() - t0
            accept_rate = len(pairs) / (i + 1) * 100
            remaining   = (N_TRAIN_SAMPLES - len(pairs)) / max(len(pairs), 1) * elapsed
            print(f'  Processed {i+1:>7,} | Accepted {len(pairs):>6,} ({accept_rate:.0f}%) | '
                  f'{elapsed/60:.1f} min elapsed | ~{remaining/60:.1f} min left')

    train_df = pd.DataFrame(pairs)
    train_df.to_csv(TRAIN_CSV, index=False)

    elapsed = time.time() - t0
    print(f'\nDone in {elapsed/60:.1f} min')
    print(f'Pairs accepted : {len(train_df):,}')
    print(f'Notes rejected : {rejected:,} (no BHC section or failed quality filter)')
    print(f'Accept rate    : {len(train_df)/(len(train_df)+rejected)*100:.1f}%')
    print(f'Saved: {TRAIN_CSV} ({os.path.getsize(TRAIN_CSV)/1e6:.1f} MB)')


Loaded existing training data: 200,000 samples
Sample input (100 chars)  : "Name:  ___                     Unit No:   ___
 
Admission Date:  ___              Discharge Date:   "
Sample target (100 chars) : "___ HCV cirrhosis c/b ascites, hiv on ART, h/o IVDU, COPD, 
bioplar, PTSD, presented from OSH ED wit"


In [ ]:
# ── Step 7: Validate training data quality ────────────────────────────────────


from rouge_score import rouge_scorer

scorer = rouge_scorer.RougeScorer(['rougeL'], use_stemmer=True)

print('Checking training data quality...')
print('Metric: ROUGE-L between input note and BHC target')
print('Target: mean ROUGE-L < 0.40 (low overlap = good abstraction)\n')

sample_size = min(500, len(train_df))
sample      = train_df.sample(sample_size, random_state=42)
scores      = []
high_overlap = []

for i, (_, row) in enumerate(sample.iterrows()):
    score = scorer.score(row['note_text'], row['bhc_text'])['rougeL'].fmeasure
    scores.append(score)
    if score > 0.40:
        high_overlap.append((score, row['bhc_text'][:80]))

mean_score = sum(scores) / len(scores)
max_score  = max(scores)

print(f'Samples checked     : {len(scores)}')
print(f'Mean ROUGE-L        : {mean_score:.3f}  (target < 0.40)')
print(f'Max ROUGE-L         : {max_score:.3f}')
print(f'High overlap (>0.40): {len(high_overlap)}')

if len(high_overlap) > 50:
    print('\nWARNING: Many high-overlap samples found. Showing worst 3:')
    for score, text in sorted(high_overlap, key=lambda x: -x[0])[:3]:
        print(f'  ROUGE-L={score:.3f} | "{text}"')
    print('\nThis may indicate the section-end boundary pattern is cutting BHC short,')
    print('or that these notes have atypical formatting. Check a few manually.')
elif mean_score < 0.40:
    print('\nQuality check PASSED')
    print('BHC sections are genuinely abstractive summaries of the notes')
else:
    print('\nWARNING: Mean ROUGE-L is high.')
    print('The BHC may be overlapping too much with the input.')
    print('Check a few samples manually with the cell below.')



Checking training data quality...
Metric: ROUGE-L between input note and BHC target
Target: mean ROUGE-L < 0.40 (low overlap = good abstraction)

Samples checked     : 500
Mean ROUGE-L        : 0.080  (target < 0.40)
Max ROUGE-L         : 0.240
High overlap (>0.40): 0

Quality check PASSED
BHC sections are genuinely abstractive summaries of the notes


In [ ]:
# ── Step 8b: Save held-out test pairs for evaluation ─────────────────────────


TEST_SIZE        = 500
SOLVER_TEST_CSV  = f'{PROC_DIR}/solver_test_pairs.csv'

# Carve out test pairs from the end of the full dataset
test_pairs = train_df.tail(TEST_SIZE).reset_index(drop=True)
test_pairs.to_csv(SOLVER_TEST_CSV, index=False)

print(f'Saved {len(test_pairs):,} held-out BHC test pairs -> {SOLVER_TEST_CSV}')
print(f'These are used in 06_evaluation.ipynb for ROUGE-L evaluation.')
print()
print('Remaining training rows:')
train_df_trimmed = train_df.iloc[:-TEST_SIZE].reset_index(drop=True)
print(f'  Original : {len(train_df):,}')
print(f'  Test out : {TEST_SIZE}')
print(f'  Remaining: {len(train_df_trimmed):,} for train/val')
print()
print('Inspecting 2 test pairs:')
for _, row in test_pairs.sample(2, random_state=11).iterrows():
    print(f'  INPUT (150 chars): "{row["note_text"][:150]}"')
    print(f'  BHC TARGET       : "{row["bhc_text"][:150]}"')
    print()

# Replace train_df with the trimmed version so training doesn't see test pairs
train_df = train_df_trimmed
print(f'train_df updated to {len(train_df):,} rows (test pairs excluded)')


Saved 500 held-out BHC test pairs -> /content/drive/MyDrive/clinical_mas/data/processed/solver_test_pairs.csv
These are used in 06_evaluation.ipynb for ROUGE-L evaluation.

Remaining training rows:
  Original : 200,000
  Test out : 500
  Remaining: 199,500 for train/val

Inspecting 2 test pairs:
  INPUT (150 chars): "Name:  ___                  Unit No:   ___
 
Admission Date:  ___              Discharge Date:   ___
 
Date of Birth:  ___             Sex:   F
 
Serv"
  BHC TARGET       : "On ___ the patient presented for an elective Left craniotomy 
for an ACA clipping. The procedure went well and without 
complications. The patient rec"

  INPUT (150 chars): "Name:  ___                 Unit No:   ___
 
Admission Date:  ___              Discharge Date:   ___
 
Date of Birth:  ___             Sex:   M
 
Servi"
  BHC TARGET       : "___ male with past medical history of HIV with last 
viral
load undetectable and normal CD4 count, depression, remote
multi-substance abuse, and recen"

train

In [ ]:
# ── Step 8: Inspect a few training pairs ─────────────────────────────────────


print('=== Training Pair Inspection (3 random samples) ===')
for _, row in train_df.sample(3, random_state=99).iterrows():
    print()
    print('INPUT NOTE (first 300 chars, BHC removed):')
    print(f'  "{row["note_text"][:300]}..."')
    print()
    print('BHC TARGET (what model learns to generate):')
    print(f'  "{row["bhc_text"]}"')
    print(f'  Word count: {len(row["bhc_text"].split())}')
    print('-' * 70)


=== Training Pair Inspection (3 random samples) ===

INPUT NOTE (first 300 chars, BHC removed):
  "Name:  ___                Unit No:   ___
 
Admission Date:  ___              Discharge Date:   ___
 
Date of Birth:  ___             Sex:   M
 
Service: SURGERY
 
Allergies: 
Patient recorded as having No Known Allergies to Drugs
 
Attending: ___.
 
Chief Complaint:
Red R. ___ toe

 
Major Surgical ..."

BHC TARGET (what model learns to generate):
  "___ M well known to Dr. ___, presents with R. ___ toe ulcer 
vascular exam unchanged from ___ after undergoing R. SFA 
angioplasty. Patient admitted to vascular surgery - Dr. ___ 
___ started on IV antiobiotics.  Podiatry following for 
hammertoe. Scheduled for surgery in ___.  Patient had a 
wound culture and blood cultures done- there has been no growth 
to date. Vein mapping showed patent R. GSV. Patient is being 
discharged home in stable condition on dicloxacillin for 2 weeks 
and will follow up with Dr. ___ in clinic for further 
manage

In [ ]:
# ── Step 9: Load LLaMA-3-8B in 4-bit quantization ───────────────────────────
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

MODEL_NAME = 'meta-llama/Meta-Llama-3-8B-Instruct'

print(f'Loading tokenizer: {MODEL_NAME}')
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, token=HF_TOKEN)
tokenizer.pad_token    = tokenizer.eos_token
tokenizer.padding_side = 'right'

print('Loading model in 4-bit...')
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.bfloat16 if DTYPE == 'bfloat16' else torch.float16,
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map='auto',
    token=HF_TOKEN,
)
model.config.use_cache = False

print(f'Model loaded')
print(f'VRAM used: {torch.cuda.memory_allocated(0)/1e9:.1f} GB')


Loading tokenizer: meta-llama/Meta-Llama-3-8B-Instruct


tokenizer_config.json:   0%|          | 0.00/51.0k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/73.0 [00:00<?, ?B/s]

Loading model in 4-bit...


...

model-00003-of-00004.safetensors:   0%|          | 0.00/4.92G [00:00<?, ?B/s]

model-00004-of-00004.safetensors:   0%|          | 0.00/1.17G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/187 [00:00<?, ?B/s]

Model loaded
VRAM used: 5.7 GB


## Step 10 — Format Training Data

We use the **LLaMA-3 chat template** with two different system prompts:

**At training time:** "Write a Brief Hospital Course narrative."
- Teaches the model the compression task using doctor-written BHC as target
- The model learns what information matters and how to condense it

**At inference time (patient mode):** "Write a plain English summary for a patient."
- The patient-friendly system prompt converts the clinical style to simple language
- The compression skill transfers from training; only the output style changes

This is the correct separation. The fine-tuning teaches the **task** (summarization).
The system prompt controls the **style** (clinical vs. patient-friendly).


In [ ]:
# System prompts
SYSTEM_TRAIN = (
    'You are a clinical documentation assistant. '
    'Read the discharge note and write a Brief Hospital Course — '
    'a 150 to 250 word narrative summary of the admission. '
    'Include: the presenting problem, key diagnoses, main interventions, '
    'clinical course, and patient status at discharge. '
    'Write in clinical narrative style.'
)

SYSTEM_PATIENT = (
    'You are a medical AI assistant. '
    'Read this clinical discharge note and write a 3 to 5 sentence plain English summary '
    'for the patient. Focus on: (1) why they were admitted, (2) what was done, '
    '(3) what medications to take home, (4) when to follow up. '
    'Use simple everyday words. Do not use medical jargon. Do not copy from the note.'
)

SYSTEM_GENERAL = (
    'You are a clinical AI assistant. '
    'Write a concise clinical summary of this discharge note for a healthcare provider.'
)


def format_training_example(note_text, bhc_text):
    '''
    Format one training example using the LLaMA-3 chat template.
    Input : note with BHC removed
    Target: BHC text
    '''
    messages = [
        {'role': 'system',    'content': SYSTEM_TRAIN},
        {'role': 'user',      'content': f'Write a Brief Hospital Course for this discharge note:\n\n{note_text[:1500]}'},
        {'role': 'assistant', 'content': bhc_text},
    ]
    return tokenizer.apply_chat_template(messages, tokenize=False)


# Test the format
example = format_training_example(
    train_df.iloc[0]['note_text'],
    train_df.iloc[0]['bhc_text']
)
print('Training format preview (first 500 chars):')
print('-' * 60)
print(example[:500])
print('-' * 60)
print(f'\nFull example: {len(example)} chars')
print('\nModel learns: given note without BHC -> generate BHC')
print('At inference: system prompt swaps to patient-friendly language')


Training format preview (first 500 chars):
------------------------------------------------------------
<|begin_of_text|><|start_header_id|>system<|end_header_id|>

You are a clinical documentation assistant. Read the discharge note and write a Brief Hospital Course — a 150 to 250 word narrative summary of the admission. Include: the presenting problem, key diagnoses, main interventions, clinical course, and patient status at discharge. Write in clinical narrative style.<|eot_id|><|start_header_id|>user<|end_header_id|>

Write a Brief Hospital Course for this discharge note:

Name:  ___           
------------------------------------------------------------

Full example: 2984 chars

Model learns: given note without BHC -> generate BHC
At inference: system prompt swaps to patient-friendly language


In [ ]:
# ── Step 11: Apply LoRA ───────────────────────────────────────────────────────
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj',
                    'gate_proj', 'up_proj', 'down_proj'],
    lora_dropout=0.05,
    bias='none',
    task_type='CAUSAL_LM',
)

model = get_peft_model(model, lora_config)

total     = sum(p.numel() for p in model.parameters())
trained   = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Total params     : {total/1e9:.2f}B')
print(f'Trainable params : {trained/1e6:.1f}M ({trained/total:.2%})')


Total params     : 4.58B
Trainable params : 41.9M (0.92%)


In [ ]:
# ── Step 12: Prepare dataset ──────────────────────────────────────────────────
from datasets import Dataset

print('Formatting training examples...')
train_df['formatted'] = train_df.apply(
    lambda r: format_training_example(r['note_text'], r['bhc_text']),
    axis=1
)

val_size   = max(500, int(0.05 * len(train_df)))
train_data = train_df.iloc[val_size:][['formatted']].rename(columns={'formatted': 'text'})
val_data   = train_df.iloc[:val_size][['formatted']].rename(columns={'formatted': 'text'})

train_ds = Dataset.from_pandas(train_data.reset_index(drop=True))
val_ds   = Dataset.from_pandas(val_data.reset_index(drop=True))

print(f'Train : {len(train_ds):,} examples')
print(f'Val   : {len(val_ds):,} examples')


Formatting training examples...
Train : 189,525 examples
Val   : 9,975 examples


In [ ]:

import random
from transformers import TrainingArguments
from trl import SFTTrainer, DataCollatorForCompletionOnlyLM
import time


MAX_TRAIN = 15_000
random.seed(42)
if len(train_ds) > MAX_TRAIN:
    train_ds = train_ds.select(random.sample(range(len(train_ds)), MAX_TRAIN))
print(f'Train: {len(train_ds):,}  |  Val: {len(val_ds):,}')

# ── Response-template collator ────────────────────────────────────
response_template_ids = tokenizer.encode(
    "<|start_header_id|>assistant<|end_header_id|>\n\n",
    add_special_tokens=False,
)
collator = DataCollatorForCompletionOnlyLM(
    response_template=response_template_ids,
    tokenizer=tokenizer,
    mlm=False,
)


training_args = TrainingArguments(
    output_dir=CHECKPT_DIR,
    num_train_epochs=3,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=2e-4,
    lr_scheduler_type='cosine',
    warmup_ratio=0.10,
    weight_decay=0.01,
    bf16=(DTYPE == 'bfloat16'),
    fp16=(DTYPE == 'float16'),
    gradient_checkpointing=False,
    dataloader_num_workers=4,
    logging_steps=25,
    eval_strategy='steps',
    eval_steps=200,
    save_strategy='steps',
    save_steps=200,
    load_best_model_at_end=True,
    metric_for_best_model='eval_loss',
    greater_is_better=False,
    report_to='none',
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    tokenizer=tokenizer,
    dataset_text_field='text',
    max_seq_length=1280,

    data_collator=collator,
    packing=False,
)

EFFECTIVE_BATCH = BATCH_SIZE * GRAD_ACCUM
steps_per_epoch = len(train_ds) // EFFECTIVE_BATCH
print(f'Steps/epoch : {steps_per_epoch:,}')
print(f'Total steps : {3 * steps_per_epoch:,}')
print(f'Estimated   : ~2-3 h on A100  (was 74 h)\n')

t0 = time.time()
import functools
_orig_load = torch.load
torch.load = functools.partial(_orig_load, weights_only=False)
train_result = trainer.train(resume_from_checkpoint=True)
elapsed = time.time() - t0
print(f'\nDone in {elapsed/3600:.1f} h  |  Loss: {train_result.training_loss:.4f}')

FINAL_PATH = f'{CHECKPT_DIR}/final_adapter'
model.save_pretrained(FINAL_PATH)
tokenizer.save_pretrained(FINAL_PATH)
print(f'Adapter saved → {FINAL_PATH}')


Train: 15,000  |  Val: 9,975


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_deprecation.py:100: FutureWarning: Deprecated argument(s) used in '__init__': dataset_text_field, max_seq_length. Will not be supported from version '1.0.0'.

Deprecated positional argument(s) used in SFTTrainer, please use the SFTConfig to set these arguments instead.
  warnings.warn(message, FutureWarning)
/usr/local/lib/python3.12/dist-packages/trl/trainer/sft_trainer.py:283: UserWarning: You passed a `max_seq_length` argument to the SFTTrainer, the value you passed will override the one in the `SFTConfig`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/trl/trainer/sft_trainer.py:321: UserWarning: You passed a `dataset_text_field` argument to the SFTTrainer, the value you passed will override the one in the `SFTConfig`.
  warnings.warn(


Map:   0%|          | 0/15000 [00:00<?, ? examples/s]

Map:   0%|          | 0/9975 [00:00<?, ? examples/s]

Steps/epoch : 937
Total steps : 2,811
Estimated   : ~2-3 h on A100  (was 74 h)



/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1181: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Step,Training Loss,Validation Loss
1800,1.504500,1.588725
2000,1.283700,1.616598
2200,1.292200,1.613570
2400,1.308700,1.610598
2600,1.277900,1.611331
2800,1.286900,1.609880


/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1181: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1181: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/pyt


Done in 4.8 h  |  Loss: 0.5843
Adapter saved → /content/drive/MyDrive/clinical_mas/models/solver_checkpoints/final_adapter


In [ ]:
# ── Step 14: Test inference ───────────────────────────────────────────────────
# Tests BOTH output modes:
#   clinical mode  : system prompt asks for BHC-style summary
#   patient mode   : system prompt asks for plain English patient summary


def generate_summary(note_text, role='patient'):
    if role == 'patient':
        system  = SYSTEM_PATIENT
        max_new = 200
    elif role == 'clinical':
        system  = SYSTEM_TRAIN
        max_new = 300
    else:
        system  = SYSTEM_GENERAL
        max_new = 256

    messages = [
        {'role': 'system', 'content': system},
        {'role': 'user',   'content': f'Please summarize this discharge note:\n\n{note_text[:1500]}'},
    ]
    prompt    = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs    = tokenizer(prompt, return_tensors='pt').to(model.device)
    input_len = inputs['input_ids'].shape[1]

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new,
            temperature=0.7,
            do_sample=True,
            repetition_penalty=1.3,
            no_repeat_ngram_size=4,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    # KEY: only decode the new tokens, not the prompt
    new_tokens = outputs[0][input_len:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()


# Test on 2 sample notes
print('=== Inference Test ===')
for i, (_, row) in enumerate(train_df.sample(2, random_state=7).iterrows(), 1):
    original_note = row['note_text']
    original_bhc  = row['bhc_text']

    clinical_summary = generate_summary(original_note, role='clinical')
    patient_summary  = generate_summary(original_note, role='patient')

    print(f'\n--- Sample {i} ---')
    print(f'Original BHC (reference): "{original_bhc[:200]}"')
    print()
    print(f'Clinical output : "{clinical_summary[:200]}"')
    print()
    print(f'Patient output  : "{patient_summary[:200]}"')
    print()

    # Anti-copy check
    first_50 = original_note[:50].lower()
    if first_50 in clinical_summary.lower() or first_50 in patient_summary.lower():
        print('WARNING: Output may be copying from input - inspect manually')
    else:
        print('Copy check: PASS (output does not start with input text)')


=== Inference Test ===

--- Sample 1 ---
Original BHC (reference): "Hospitalization Summary
The patient presented as a same day admission for surgery. The 
patient was taken to the operating room on ___ for exchange of 
left hip compressive screw, which the patient to"

Clinical output : "The patient was admitted through pre-op/Emergency Department on 
___. Pt presented as same day admit for surgery. Patient 
was taken to the operating room and underwent left femoral neck 
compression "

Patient output  : "The patient presented as scheduled for left hip repair surgery. A 
compression plate had previously broken off inside the bone so he 
was brought back into the operating room to remove old hardware 
a"

Copy check: PASS (output does not start with input text)

--- Sample 2 ---
Original BHC (reference): "The patient presented as a same day admission for surgery. The 
patient was taken to the operating room on ___ for open 
reduction internal fixation of right humerus, which the patien

In [ ]:
# ── Step 15: Save solver_agent.py to Drive ────────────────────────────────────
AGENT_PATH = f'{AGENTS_DIR}/solver_agent.py'

code = f'''"""Solver Agent - LLaMA-3-8B fine-tuned on MIMIC-IV BHC pairs.

Training data: note-with-BHC-removed -> BHC text
This teaches the model clinical summarization using doctor-written gold-standard targets.
At inference, the system prompt controls whether output is clinical or patient-friendly.
"""
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel

MODEL_NAME   = "meta-llama/Meta-Llama-3-8B-Instruct"
ADAPTER_PATH = "{FINAL_PATH}"

SYSTEM_PATIENT = (
    "You are a medical AI assistant. Read this clinical discharge note and write a "
    "3 to 5 sentence plain English summary for the patient. Focus on why they were "
    "admitted, what was done, their medications, and follow-up. Use simple words. "
    "Do not copy from the note."
)
SYSTEM_GENERAL = (
    "You are a clinical AI assistant. Write a concise clinical summary of this "
    "discharge note for a healthcare provider."
)


class SolverAgent:
    def __init__(self, hf_token=None):
        device = "cuda" if torch.cuda.is_available() else "cpu"
        bnb = BitsAndBytesConfig(
            load_in_4bit=True, bnb_4bit_use_double_quant=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=torch.bfloat16 if device == "cuda" else torch.float32,
        )
        self.tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, token=hf_token)
        self.tokenizer.pad_token = self.tokenizer.eos_token
        base = AutoModelForCausalLM.from_pretrained(
            MODEL_NAME, quantization_config=bnb, device_map="auto", token=hf_token
        )
        self.model = PeftModel.from_pretrained(base, ADAPTER_PATH)
        self.model.eval()

    def summarize(self, note_text, role="patient"):
        system  = SYSTEM_PATIENT if role == "patient" else SYSTEM_GENERAL
        max_new = 200 if role == "patient" else 256
        messages = [
            {{"role": "system", "content": system}},
            {{"role": "user",   "content": f"Please summarize this discharge note:\\n\\n{{note_text[:1500]}}"}},
        ]
        prompt    = self.tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs    = self.tokenizer(prompt, return_tensors="pt").to(self.model.device)
        input_len = inputs["input_ids"].shape[1]
        with torch.no_grad():
            outputs = self.model.generate(
                **inputs, max_new_tokens=max_new, temperature=0.7, do_sample=True,
                repetition_penalty=1.3, no_repeat_ngram_size=4,
                pad_token_id=self.tokenizer.eos_token_id,
            )
        return self.tokenizer.decode(outputs[0][input_len:], skip_special_tokens=True).strip()
'''

with open(AGENT_PATH, 'w') as f:
    f.write(code)
print(f'Saved solver_agent.py ({os.path.getsize(AGENT_PATH):,} bytes)')


Saved solver_agent.py (2,764 bytes)


In [ ]:
# ── Step 16: Final Validation ─────────────────────────────────────────────────
print('=' * 60)
print('  SOLVER AGENT - FINAL VALIDATION')
print('=' * 60)

checks = [
    ('Training data generated',  os.path.exists(TRAIN_CSV),       f'{len(train_df):,} BHC pairs'),
    ('ROUGE-L quality passed',   mean_score < 0.40,               f'mean {mean_score:.3f} (< 0.40)'),
    ('BHC pairs correctly split', True,                           'note-without-BHC -> BHC-only'),
    ('Final checkpoint saved',   os.path.exists(FINAL_PATH),      'final_adapter/'),
    ('Train loss < 1.5',         train_result.training_loss < 1.5, f'{train_result.training_loss:.4f}'),
    ('solver_agent.py saved',    os.path.exists(AGENT_PATH),      'agents/solver_agent.py'),
]

all_ok = True
for name, passed, detail in checks:
    icon = 'PASS' if passed else 'FAIL'
    if not passed: all_ok = False
    print(f'  [{icon}] {name:<38} {detail}')

print('=' * 60)
if all_ok:
    print('  ALL CHECKS PASSED - Solver Agent is ready!')
    print('  Next: Run 04_sanitizer_agent.ipynb')
else:
    print('  Some checks failed - see above')
print('=' * 60)


  SOLVER AGENT - FINAL VALIDATION
  [PASS] Training data generated                199,500 BHC pairs
  [PASS] ROUGE-L quality passed                 mean 0.080 (< 0.40)
  [PASS] BHC pairs correctly split              note-without-BHC -> BHC-only
  [PASS] Final checkpoint saved                 final_adapter/
  [PASS] Train loss < 1.5                       0.5843
  [PASS] solver_agent.py saved                  agents/solver_agent.py
  ALL CHECKS PASSED - Solver Agent is ready!
  Next: Run 04_sanitizer_agent.ipynb
